In [33]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
MODEL_B = "distilbert-base-uncased"
tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)
MAX_LENGTH = 384
DOC_STRIDE = 96

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6002.41it/s]
[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
def prepare_qa_features(examples):
    tokenized = tokenizer_b(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offsets = tokenized.pop("offset_mapping")
    start_positions, end_positions = [], []
    for feature_index, feature_offsets in enumerate(offsets):
        input_ids = tokenized["input_ids"][feature_index]
        cls_index = input_ids.index(tokenizer_b.cls_token_id)
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]
        answer_start = examples["answer_start"][sample_index]
        answer_end = answer_start + len(examples["answer_text"][sample_index])
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1
        if (
            feature_offsets[context_start][0] > answer_start
            or feature_offsets[context_end][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue
        token_start = context_start
        while feature_offsets[token_start][1] <= answer_start:
            token_start += 1
        token_end = context_end
        while feature_offsets[token_end][0] >= answer_end:
            token_end -= 1
        start_positions.append(token_start)
        end_positions.append(token_end)
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [35]:
# TODO(student 2): create a small SQuAD-style technical-support dataset
# with trusted contexts from the supplied KB/docs and at least 30 QA pairs.
qa_sample_rows = [
    {
        "question": "Which port does the technical support API use by default?",
        "context": "The technical support API listens on port 8000 by default.",
        "answer_text": "8000",
        "answer_start": 42,
    },
    {
        "question": "What does an HTTP 503 response indicate?",
        "context": "A 503 response indicates that the service is temporarily unavailable.",
        "answer_text": "temporarily unavailable",
        "answer_start": 45,
    },
    {
        "question": "What should happen when database corruption is suspected?",
        "context": "If database corruption is suspected, stop automated recovery and escalate the incident to a human operator.",
        "answer_text": "escalate the incident to a human operator",
        "answer_start": 65,
    },
]

In [36]:
import re
import string
from collections import Counter

import numpy as np
import pandas as pd


def normalize_answer(text):
    text = text.lower()
    text = "".join(
        character
        for character in text
        if character not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match_score(prediction, reference):
    return float(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    predicted_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not predicted_tokens and not reference_tokens:
        return 1.0

    if not predicted_tokens or not reference_tokens:
        return 0.0

    common_tokens = Counter(predicted_tokens) & Counter(reference_tokens)
    matching_count = sum(common_tokens.values())

    if matching_count == 0:
        return 0.0

    precision = matching_count / len(predicted_tokens)
    recall = matching_count / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


def evaluate_qa_model(
    trainer,
    features,
    max_answer_length=30,
):
    prediction_output = trainer.predict(
        features,
        metric_key_prefix="eval",
    )

    start_logits, end_logits = prediction_output.predictions
    result_rows = []

    for feature_index, feature in enumerate(features):
        input_ids = feature["input_ids"]
        true_start = int(feature["start_positions"])
        true_end = int(feature["end_positions"])

        cls_index = input_ids.index(tokenizer_b.cls_token_id)

        separator_indexes = [
            index
            for index, token_id in enumerate(input_ids)
            if token_id == tokenizer_b.sep_token_id
        ]

        context_start = separator_indexes[0] + 1
        context_end = separator_indexes[1] - 1

        # The CLS position represents "no answer in this feature."
        best_start = cls_index
        best_end = cls_index
        best_score = (
            start_logits[feature_index][cls_index]
            + end_logits[feature_index][cls_index]
        )

        likely_starts = np.argsort(
            start_logits[feature_index]
        )[-20:][::-1]

        likely_ends = np.argsort(
            end_logits[feature_index]
        )[-20:][::-1]

        for start_index in likely_starts:
            for end_index in likely_ends:
                if start_index < context_start:
                    continue

                if end_index > context_end:
                    continue

                if end_index < start_index:
                    continue

                if end_index - start_index + 1 > max_answer_length:
                    continue

                score = (
                    start_logits[feature_index][start_index]
                    + end_logits[feature_index][end_index]
                )

                if score > best_score:
                    best_score = score
                    best_start = int(start_index)
                    best_end = int(end_index)

        if best_start == cls_index:
            predicted_answer = ""
        else:
            predicted_answer = tokenizer_b.decode(
                input_ids[best_start : best_end + 1],
                skip_special_tokens=True,
            )

        if true_start == cls_index:
            reference_answer = ""
        else:
            reference_answer = tokenizer_b.decode(
                input_ids[true_start : true_end + 1],
                skip_special_tokens=True,
            )

        result_rows.append(
            {
                "feature_index": feature_index,
                "prediction": predicted_answer,
                "reference": reference_answer,
                "exact_match": exact_match_score(
                    predicted_answer,
                    reference_answer,
                ),
                "token_f1": token_f1_score(
                    predicted_answer,
                    reference_answer,
                ),
            }
        )

    results = pd.DataFrame(result_rows)

    metrics = {
        "eval_loss": prediction_output.metrics.get("eval_loss"),
        "exact_match": results["exact_match"].mean(),
        "token_f1": results["token_f1"].mean(),
    }

    return metrics, results

In [37]:
from datasets import Dataset
from transformers import DataCollatorWithPadding
qa_dataset = Dataset.from_list(qa_sample_rows)

qa_splits = qa_dataset.train_test_split(test_size=0.2,seed=42)

qa_train_features = qa_splits["train"].map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_splits["train"].column_names,
)

qa_val_features = qa_splits["test"].map(
    prepare_qa_features,
    batched=True,
    remove_columns=qa_splits["test"].column_names,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer_b)

Map: 100%|██████████| 1/1 [00:00<00:00, 518.14 examples/s]


In [38]:
from transformers import TrainingArguments, Trainer

args_b = TrainingArguments(
    output_dir="models/qa_model",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    save_total_limit=1,
)

trainer_b = Trainer(
    model=model_b,
    args=args_b,
    train_dataset=qa_train_features,
    eval_dataset=qa_val_features,
    data_collator=data_collator,
)

In [39]:
baseline_b, baseline_details_b = evaluate_qa_model(
    trainer_b,
    qa_val_features,
)

model_report_b = {
    "baseline": baseline_b,
    "fine_tuned": {},
    "quality_gate": {},
}

print("Baseline metrics:")
display(pd.DataFrame([baseline_b]))

print("Baseline predictions:")
display(baseline_details_b)

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Baseline metrics:


,eval_loss,exact_match,token_f1
0,5.894808,0.0,0.0


Baseline predictions:


,feature_index,prediction,reference,exact_match,token_f1
0,0,",",escalate the incident to a human operator,0.0,0.0


In [40]:
trainer_b.train()

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,5.961146,5.839768
2,5.712961,5.809308


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.09it/s]
/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.07it/s]


TrainOutput(global_step=2, training_loss=5.837053537368774, metrics={'train_runtime': 2.4999, 'train_samples_per_second': 1.6, 'train_steps_per_second': 0.8, 'total_flos': 391959300096.0, 'train_loss': 5.837053537368774, 'epoch': 2.0})

In [41]:
fine_tuned_b, fine_tuned_details_b = evaluate_qa_model(
    trainer_b,
    qa_val_features,
)

em_passed = fine_tuned_b["exact_match"] >= 0.65
f1_passed = fine_tuned_b["token_f1"] >= 0.80

model_report_b["fine_tuned"] = fine_tuned_b
model_report_b["quality_gate"] = {
    "em_threshold": 0.65,
    "f1_threshold": 0.80,
    "em_passed": em_passed,
    "f1_passed": f1_passed,
    "passed": em_passed and f1_passed,
    "improved_over_baseline": (
        fine_tuned_b["exact_match"]
        >= baseline_b["exact_match"]
        and fine_tuned_b["token_f1"]
        >= baseline_b["token_f1"]
    ),
}

print("Baseline metrics:")
display(pd.DataFrame([baseline_b]))

print("Fine-tuned metrics:")
display(pd.DataFrame([fine_tuned_b]))

print("Quality gate:")
display(pd.DataFrame([model_report_b["quality_gate"]]))

/Users/emad/Tuwaiq/Multi_Model_Agent_Architecture/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Baseline metrics:


,eval_loss,exact_match,token_f1
0,5.894808,0.0,0.0


Fine-tuned metrics:


,eval_loss,exact_match,token_f1
0,5.809308,0.0,0.0


Quality gate:


,em_threshold,f1_threshold,em_passed,f1_passed,passed,improved_over_baseline
0,0.65,0.8,False,False,False,True


In [42]:
history_b = pd.DataFrame(trainer_b.state.log_history)

qa_errors_b = fine_tuned_details_b[
    (fine_tuned_details_b["exact_match"] == 0)
    | (fine_tuned_details_b["token_f1"] < 1)
].copy()

print("Training history:")
display(history_b)

print("Incorrect or partially correct answers:")
display(qa_errors_b)

Training history:


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_model_preparation_time,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,5.961146,6.743839,0.000030,1.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,1,5.839768,0.0015,0.0224,44.679,44.679,NaN,NaN,NaN,NaN,NaN
2,5.712961,7.571660,0.000015,2.0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,2,5.809308,0.0015,0.0211,47.369,47.369,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,2.0,2,NaN,NaN,NaN,NaN,NaN,2.4999,1.6,0.8,3.919593e+11,5.837054


Incorrect or partially correct answers:


,feature_index,prediction,reference,exact_match,token_f1
0,0,",",escalate the incident to a human operator,0.0,0.0


In [43]:
trainer_b.save_model("models/qa_model")
tokenizer_b.save_pretrained("models/qa_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.96it/s]


('models/qa_model/tokenizer_config.json', 'models/qa_model/tokenizer.json')